In [ ]:
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier


# -----------------------------
# 1. 데이터
# -----------------------------
data = pd.DataFrame({
    "고객": ["A","B","C","D"],
    "사용기간":[1,2,8,9],
    "이탈":[1,1,0,0]
})

print(data)


# -----------------------------
# 2. X / y
# -----------------------------
X = data[["사용기간"]]
y = data["이탈"]


# -----------------------------
# 3. 초기 가중치
# -----------------------------

# 4. 초기 가중치 만들기
# weights = np.ones(len(data)) / len(data)
# 의미
# step1: np.ones(4)
# [1, 1, 1, 1]
# step2: / 4
# [0.25, 0.25, 0.25, 0.25]

# 👉 의미:

# 모든 고객을 똑같이 중요하게 봄

weights = np.ones(
    len(data)
) / len(data)

print("\n초기 가중치")

for i in range(len(data)):
    print(
        data["고객"][i],
        round(weights[i],3)
    )


# -----------------------------
# 4. 첫 번째 트리
# -----------------------------
# “복잡한 모델 금지, 약한 모델만 사용”
model1 = DecisionTreeClassifier(
    max_depth=1,
    random_state=42
)
#사용기간 <= 1.5 ?

model1.fit(
    X,
    y,
    sample_weight=weights
)



# -----------------------------
# 5. 예측
# -----------------------------
pred = model1.predict(X)

# | 고객 | 실제 | 예측 |
# | -- | -- | -- |
# | A  | 1  | 1  |
# | B  | 1  | 0  |
# | C  | 0  | 0  |
# | D  | 0  | 0  |



data["예측"] = pred

print("\n첫 번째 예측")

print(
    data[
        ["고객","이탈","예측"]
    ]
)


# -----------------------------
# 6. 틀린 데이터 찾기
# -----------------------------
wrong = (
    y
    !=
    pred
)

data["틀림"] = wrong

print("\n틀린 데이터")

print(
    data[
        ["고객","틀림"]
    ]
)


# -----------------------------
# 7. 오차 계산
# -----------------------------
error = np.sum(
    weights[wrong]
)

print("\n오차율")

print(
    round(
        error,
        3
    )
)


# -----------------------------
# 8. 모델 점수(alpha)
# -----------------------------
alpha = 0.5*np.log(
    (1-error)
    /
    error
)

print("\n모델 점수")

print(
    round(
        alpha,
        3
    )
)


# -----------------------------
# 9. 가중치 수정
# -----------------------------
weights = np.where(
    wrong,
    weights*np.exp(alpha),
    weights*np.exp(-alpha)
)


# 정규화
weights /= weights.sum()


# -----------------------------
# 10. 결과
# -----------------------------
data["새가중치"] = np.round(
    weights,
    3
)

print("\n가중치 변경")

print(
    data[
        [
            "고객",
            "틀림",
            "새가중치"
        ]
    ]
)

  고객  사용기간  이탈
0  A     1   1
1  B     2   1
2  C     8   0
3  D     9   0

초기 가중치
A 0.25
B 0.25
C 0.25
D 0.25

첫 번째 예측
  고객  이탈  예측
0  A   1   1
1  B   1   1
2  C   0   0
3  D   0   0

틀린 데이터
  고객     틀림
0  A  False
1  B  False
2  C  False
3  D  False

오차율
0.0

모델 점수
inf

가중치 변경
  고객     틀림  새가중치
0  A  False   NaN
1  B  False   NaN
2  C  False   NaN
3  D  False   NaN


/tmp/ipykernel_14779/1916064705.py:114: RuntimeWarning: divide by zero encountered in scalar divide
  (1-error)
/tmp/ipykernel_14779/1916064705.py:140: RuntimeWarning: invalid value encountered in divide
  weights /= weights.sum()
